# RISE

输出 `results/rise/<model>/<设定>/<image_id>.npy`（`[K, H, W]`）和 `figures/rise_<model>.png`。先运行初始化格；其余各格只依赖它。

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image

from fmspace.analysis import l2, pc_scorer, pca, rho
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import load_encoder
from fmspace.paths import DATA_DIR, FEATURE_DIR, FIGURE_DIR, RISE_DIR
from fmspace.plots import saliency_figure
from fmspace.rise import blurred, fov_mask, load_image, rise

MODEL = "medsiglip"
COMPONENTS = {1: "eye_dr_level", 2: "eye"}  # 主成分 → 定方向用的标签
N_MASKS, GRID, SEED = 1000, 8, 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

metadata = load_metadata(DATA_DIR).set_index("image_id", drop=False)
X = l2(load_features(FEATURE_DIR / f"{MODEL}.npz", metadata["image_id"]))
fitted = pca(X)
pcs = fitted.transform(X)[:, np.array(list(COMPONENTS)) - 1]
signs = np.array([np.sign(rho(pcs[:, j], metadata[label].rank())) for j, label in enumerate(COMPONENTS.values())])
scores = pd.DataFrame(pcs * signs, index=metadata.index, columns=[f"PC{k}" for k in COMPONENTS])

encode = load_encoder(MODEL, DEVICE)
score = pc_scorer(encode, fitted, list(COMPONENTS), signs)
OUT = RISE_DIR / MODEL / f"pc{'-'.join(map(str, COMPONENTS))}_grid{GRID}_n{N_MASKS}"
OUT.mkdir(parents=True, exist_ok=True)

## 复现检查

与 `features/<model>.npz` 的余弦相似度：`full` 用原图，`resized` 用缩到 512 px 的图。

In [ ]:
ids = metadata["image_id"][:8]
with torch.inference_mode():
    z = {
        "full": encode([Image.open(metadata["path"][i]).convert("RGB") for i in ids]),
        "resized": encode([load_image(metadata["path"][i]) for i in ids]),
    }
pd.DataFrame({k: (l2(v.float().cpu().numpy()) * X[metadata.index.get_indexer(ids)]).sum(axis=1) for k, v in z.items()}, index=ids)

## 跑 RISE

每个 DR 等级 1 张：PC1 得分最接近该等级中位数的图。

In [ ]:
ranked = scores.join(metadata[["patient_id", "eye", "eye_dr_level"]])
gap = (ranked["PC1"] - ranked.groupby("eye_dr_level")["PC1"].transform("median")).abs()
selected = ranked.loc[gap.groupby(ranked["eye_dr_level"]).idxmin()]

for image_id in selected.index:
    if not (OUT / f"{image_id}.npy").exists():
        image = load_image(metadata["path"][image_id])
        np.save(OUT / f"{image_id}.npy", rise(score, image, blurred(image), N_MASKS, GRID, seed=SEED))

## 画图

In [ ]:
ids = scores.loc[[p.stem for p in OUT.glob("*.npy")]].sort_values(scores.columns[0]).index
images = [load_image(metadata["path"][i]) for i in ids]
labels = [f"DR {metadata['eye_dr_level'][i]}, {metadata['eye'][i]}\n" + ", ".join(f"{c} {scores[c][i]:+.2f}" for c in scores) for i in ids]
fig = saliency_figure(images, [np.load(OUT / f"{i}.npy") for i in ids], [fov_mask(im) for im in images], labels, list(scores.columns))
fig.savefig(FIGURE_DIR / f"rise_{MODEL}.png", dpi=300, bbox_inches="tight")

## 稳定性

同一张图上两张显著图的 Spearman：`seed` 为两个随机种子之间，`fill` 为模糊填充与黑色填充之间。

In [ ]:
image = load_image(metadata["path"][scores.iloc[:, 0].idxmax()])
fov = fov_mask(image)
a = rise(score, image, blurred(image), N_MASKS, GRID, seed=0)
b = rise(score, image, blurred(image), N_MASKS, GRID, seed=1)
c = rise(score, image, Image.new("RGB", image.size), N_MASKS, GRID, seed=0)
pd.DataFrame({"seed": [rho(x[fov], y[fov]) for x, y in zip(a, b)], "fill": [rho(x[fov], y[fov]) for x, y in zip(a, c)]}, index=scores.columns)